In [ ]:
import os
import sys
import json
import math
import random
import warnings
from pathlib import Path
import types
import importlib
import itertools

import pandas as pd
import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader

from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    pipeline,
    get_linear_schedule_with_warmup,
)

from bert_score import score as bert_score
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from tqdm import tqdm

from Code.model.delphi import DelphiScorer
from Code.policy import Policy
from Code.value import Value
from Code.arguments import get_args
from Code.utils.constants import GAIN, BIAS


def require_paths(paths, step_hint):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        msg = f"Missing required path(s): {', '.join(missing)}. Complete {step_hint} first."
        try:
            assert False, msg
        except AssertionError as exc:
            try:
                raise FileNotFoundError(str(exc))
            except FileNotFoundError as err:
                print(f"WARNING: {err}")
        return False
    return True


def load_reward_module():
    if 'Code.reward' in sys.modules:
        return sys.modules['Code.reward']
    try:
        module = importlib.import_module('Code.reward')
        return module
    except Exception as exc:
        print(f"WARNING: Failed to import Code.reward directly: {exc}")
        module = types.ModuleType('Code.reward')
        code = Path('Code/reward.py').read_text()
        replacements = {
            "model_nli = RobertaForSequenceClassification.from_pretrained('alisawuffles/roberta-large-wanli').to(\"cuda:5\")": "model_nli = None",
            "tokenizer_nli = RobertaTokenizer.from_pretrained('alisawuffles/roberta-large-wanli')": "tokenizer_nli = None",
            "model_dict_answer = load_model_t5('out_dir_t5_large_answergen', cuda_devices=[2])": "model_dict_answer = None",
            "model_dict_fusion = load_model_t5('checkpoint-11000', cuda_devices=[2])": "model_dict_fusion = None",
            "to(\"cuda:5\")": "to(nli_device)",
        }
        for old, new in replacements.items():
            code = code.replace(old, new)
        module.__dict__['nli_device'] = "cuda:0" if torch.cuda.is_available() else "cpu"
        exec(code, module.__dict__)
        sys.modules['Code.reward'] = module
        return module


**Phase 1: Model Preparation**  
Purpose: Prepare pretrained components used throughout the ClarifyDelphi experiments.


In [ ]:
# STEP 1-1: Prepare T5-large base model.
model_t5_base = T5ForConditionalGeneration.from_pretrained("t5-large")
tokenizer_t5_base = T5Tokenizer.from_pretrained("t5-large")
print(model_t5_base.config)


In [ ]:
# STEP 1-2: Prepare Delphi model (psi_Delphi).
local_delphi_path = Path("large_commonsense_morality_hf")
if local_delphi_path.exists():
    delphi_model = T5ForConditionalGeneration.from_pretrained(str(local_delphi_path))
else:
    print("WARNING: large_commonsense_morality_hf/ not found. Falling back to HuggingFace: uw-hai/delphi")
    delphi_model = T5ForConditionalGeneration.from_pretrained("uw-hai/delphi")
    local_delphi_path.mkdir(parents=True, exist_ok=True)
    delphi_model.save_pretrained(str(local_delphi_path))

delphi_scorer = DelphiScorer(device_id="cuda:0" if torch.cuda.is_available() else "cpu", model="t5-large")
delphi_scorer.model = delphi_model.to(delphi_scorer.device)
print(delphi_scorer.score("lying to a friend"))


In [ ]:
# STEP 1-3: Prepare WaNLI NLI model.
model_nli = RobertaForSequenceClassification.from_pretrained("alisawuffles/roberta-large-wanli")
tokenizer_nli = RobertaTokenizer.from_pretrained("alisawuffles/roberta-large-wanli")
model_nli.eval()

sample_premise = "A person returns a lost wallet to its owner."
sample_hypothesis = "The person kept the wallet."
inputs = tokenizer_nli(sample_premise, sample_hypothesis, return_tensors="pt", truncation=True)
with torch.no_grad():
    logits = model_nli(**inputs).logits
probs = logits.softmax(dim=1).squeeze(0)
pred_label = model_nli.config.id2label[int(torch.argmax(probs))]
print({"label": pred_label, "probs": probs.tolist()})


# STEP 1-4: Note on Sentence Fusion model.
The Sentence Fusion model checkpoint (checkpoint-11000/) will be produced in Step 2-3 and loaded in Step 3-3.


**Phase 2: Supervised Training**  
Purpose: Train the supervised question, answer, and sentence fusion models needed for the reward pipeline.


In [ ]:
# STEP 2-1: Train theta_Q: Supervised question generation model.
train_path = Path("Data/delta-Clarify/train.tsv")
val_path = Path("Data/delta-Clarify/dev.tsv")

if require_paths([train_path, val_path], "Step 2-1"):
    train_df = pd.read_csv(train_path, sep="	")
    val_df = pd.read_csv(val_path, sep="	")

    tokenizer = T5Tokenizer.from_pretrained("t5-large")
    model = T5ForConditionalGeneration.from_pretrained("t5-large")

    class QGenDataset(Dataset):
        def __init__(self, df, tokenizer, max_source_len=512, max_target_len=128):
            self.inputs = df["situation"].astype(str).tolist()
            self.targets = df["question"].astype(str).tolist()
            self.tokenizer = tokenizer
            self.max_source_len = max_source_len
            self.max_target_len = max_target_len

        def __len__(self):
            return len(self.inputs)

        def __getitem__(self, idx):
            model_inputs = self.tokenizer(
                self.inputs[idx],
                max_length=self.max_source_len,
                truncation=True,
            )
            labels = self.tokenizer(
                text_target=self.targets[idx],
                max_length=self.max_target_len,
                truncation=True,
            )
            model_inputs["labels"] = labels["input_ids"]
            return model_inputs

    train_dataset = QGenDataset(train_df, tokenizer)
    val_dataset = QGenDataset(val_df, tokenizer)

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

    training_args = Seq2SeqTrainingArguments(
        output_dir="output_t5_large_qgen",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        evaluation_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        predict_with_generate=True,
        fp16=torch.cuda.is_available(),
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
    )

    trainer.train()
    trainer.save_model("output_t5_large_qgen")

    epoch_losses = [
        (log.get("epoch"), log.get("loss"))
        for log in trainer.state.log_history
        if "loss" in log and "epoch" in log
    ]
    for epoch, loss in epoch_losses:
        print(f"Epoch {epoch}: training loss={loss}")

    eval_metrics = trainer.evaluate()
    print(f"Final validation loss: {eval_metrics.get('eval_loss')}")


In [ ]:
# STEP 2-2: Train theta_A: Defeasible answer generation model.
train_path = Path("Data/delta-Clarify-silver/sc_train_questions_gpt.tsv")
val_path = Path("Data/delta-Clarify-silver/sc_dev_questions_gpt.tsv")

if require_paths([train_path, val_path], "Step 2-2"):
    train_df = pd.read_csv(train_path, sep="	")
    val_df = pd.read_csv(val_path, sep="	")

    tokenizer = T5Tokenizer.from_pretrained("t5-large")
    model = T5ForConditionalGeneration.from_pretrained("t5-large")

    def build_answer_input(df):
        return (
            "answer: "
            + df["Hypothesis"].astype(str)
            + " TYPE: "
            + df["UpdateType"].astype(str)
            + " QUESTION: "
            + df["question_davinci"].astype(str)
        )

    class AnswerGenDataset(Dataset):
        def __init__(self, df, tokenizer, max_source_len=512, max_target_len=128):
            self.inputs = build_answer_input(df).tolist()
            self.targets = df["Update"].astype(str).tolist()
            self.tokenizer = tokenizer
            self.max_source_len = max_source_len
            self.max_target_len = max_target_len

        def __len__(self):
            return len(self.inputs)

        def __getitem__(self, idx):
            model_inputs = self.tokenizer(
                self.inputs[idx],
                max_length=self.max_source_len,
                truncation=True,
            )
            labels = self.tokenizer(
                text_target=self.targets[idx],
                max_length=self.max_target_len,
                truncation=True,
            )
            model_inputs["labels"] = labels["input_ids"]
            return model_inputs

    train_dataset = AnswerGenDataset(train_df, tokenizer)
    val_dataset = AnswerGenDataset(val_df, tokenizer)

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

    training_args = Seq2SeqTrainingArguments(
        output_dir="out_dir_t5_large_answergen",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        evaluation_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        predict_with_generate=True,
        fp16=torch.cuda.is_available(),
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
    )

    trainer.train()
    trainer.save_model("out_dir_t5_large_answergen")

    epoch_losses = [
        (log.get("epoch"), log.get("loss"))
        for log in trainer.state.log_history
        if "loss" in log and "epoch" in log
    ]
    for epoch, loss in epoch_losses:
        print(f"Epoch {epoch}: training loss={loss}")

    eval_metrics = trainer.evaluate()
    print(f"Final validation loss: {eval_metrics.get('eval_loss')}")


In [ ]:
# STEP 2-3: Train Sentence Fusion model.
train_path = Path("Data/fusion/fusion_train.csv")
val_path = Path("Data/fusion/fusion_dev.csv")

if require_paths([train_path, val_path], "Step 2-3"):
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)

    tokenizer = T5Tokenizer.from_pretrained("t5-large")
    model = T5ForConditionalGeneration.from_pretrained("t5-large")

    class FusionDataset(Dataset):
        def __init__(self, df, tokenizer, max_source_len=512, max_target_len=128):
            self.inputs = ("merge: " + df["input"].astype(str)).tolist()
            self.targets = df["output"].astype(str).tolist()
            self.tokenizer = tokenizer
            self.max_source_len = max_source_len
            self.max_target_len = max_target_len

        def __len__(self):
            return len(self.inputs)

        def __getitem__(self, idx):
            model_inputs = self.tokenizer(
                self.inputs[idx],
                max_length=self.max_source_len,
                truncation=True,
            )
            labels = self.tokenizer(
                text_target=self.targets[idx],
                max_length=self.max_target_len,
                truncation=True,
            )
            model_inputs["labels"] = labels["input_ids"]
            return model_inputs

    train_dataset = FusionDataset(train_df, tokenizer)
    val_dataset = FusionDataset(val_df, tokenizer)

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

    training_args = Seq2SeqTrainingArguments(
        output_dir="checkpoint-11000",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        evaluation_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        predict_with_generate=True,
        fp16=torch.cuda.is_available(),
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
    )

    trainer.train()
    trainer.save_model("checkpoint-11000")

    epoch_losses = [
        (log.get("epoch"), log.get("loss"))
        for log in trainer.state.log_history
        if "loss" in log and "epoch" in log
    ]
    for epoch, loss in epoch_losses:
        print(f"Epoch {epoch}: training loss={loss}")

    eval_metrics = trainer.evaluate()
    print(f"Final validation loss: {eval_metrics.get('eval_loss')}")


**Phase 3: Reward Pipeline**  
Purpose: Reproduce the answer generation, fusion, and Delphi-based reward computation steps.


In [ ]:
# STEP 3-1: Generate hypothetical answers (weakener and strengthener).
required = [
    Path("out_dir_t5_large_answergen"),
    Path("output_t5_large_qgen"),
    Path("Data/delta-Clarify/test.tsv"),
]

if torch.cuda.is_available() and require_paths(required, "Steps 2-1/2-2"):
    reward_module = load_reward_module()
    reward_module.model_nli = model_nli
    reward_module.tokenizer_nli = tokenizer_nli

    model_dict_answer = reward_module.load_model_t5("out_dir_t5_large_answergen", cuda_devices=[0])
    reward_module.model_dict_answer = model_dict_answer

    policy = Policy(model_name="output_t5_large_qgen", device="cuda:0")

    test_df = pd.read_csv("Data/delta-Clarify/test.tsv", sep="	")
    sample_situations = test_df["situation"].astype(str).tolist()[:8]

    _, sample_judgements = delphi_scorer.generate_batch(sample_situations)
    policy_outputs = policy.sample(prompts=sample_situations, top_p=0.6)
    sample_questions = policy_outputs["response/text"]

    sample_bad, sample_good = reward_module.get_answers_from_model_batch_t5(
        sample_situations,
        sample_questions,
        sample_judgements,
    )

    for situation, question, bad_ans, good_ans in zip(
        sample_situations, sample_questions, sample_bad, sample_good
    ):
        print("SITUATION:", situation)
        print("QUESTION:", question)
        print("WEAKENER:", bad_ans)
        print("STRENGTHENER:", good_ans)
        print("-")
else:
    print("WARNING: CUDA is required for Policy parallelization; skipping Step 3-1.")


In [ ]:
# STEP 3-2: Apply WaNLI NLI filtering.
if "sample_situations" in globals() and "reward_module" in globals():
    def generate_raw_answers(situations, questions, judgements):
        tokenizer = reward_module.model_dict_answer["tokenizer"]
        model = reward_module.model_dict_answer["model"]
        model_inputs = []
        uts = []
        all_situations = []
        qs = []
        for jud, sit, q in zip(judgements, situations, questions):
            for ut in ["weakener", "strengthener"]:
                if not sit.endswith("."):
                    sit = sit + "."
                input_text = "answer: " + jud + " " + sit + " TYPE: " + ut + " QUESTION: " + q
                model_inputs.append(input_text)
                uts.append(ut)
                all_situations.append(sit)
                qs.append(q)
        inputs = tokenizer(model_inputs, return_tensors="pt", padding=True).to(
            reward_module.model_dict_answer["cuda_device"]
        )
        response_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            do_sample=True,
            max_length=200,
            top_p=0.6,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
        pred = tokenizer.batch_decode(response_ids, skip_special_tokens=True)
        bad_raw, good_raw = [], []
        for answer, ut in zip(pred, uts):
            if ut == "weakener":
                bad_raw.append(answer)
            else:
                good_raw.append(answer)
        return bad_raw, good_raw

    raw_bad, raw_good = generate_raw_answers(sample_situations, sample_questions, sample_judgements)
    filtered_bad, filtered_good = reward_module.get_answers_from_model_batch_t5(
        sample_situations,
        sample_questions,
        sample_judgements,
    )

    for idx, (rb, rg, fb, fg) in enumerate(zip(raw_bad, raw_good, filtered_bad, filtered_good)):
        print(f"Sample {idx}")
        print("Raw weakener:", rb)
        print("Filtered weakener:", fb)
        print("Raw strengthener:", rg)
        print("Filtered strengthener:", fg)
        print("-")
else:
    print("WARNING: Step 3-1 outputs not found; skipping Step 3-2.")


In [ ]:
# STEP 3-3: Generate updated situations via Sentence Fusion.
if "sample_situations" in globals() and require_paths([Path("checkpoint-11000")], "Step 2-3"):
    reward_module = load_reward_module()
    reward_module.model_dict_fusion = reward_module.load_model_t5("checkpoint-11000", cuda_devices=[0])

    s_ui_w = reward_module.predict_merge(sample_situations, sample_questions, sample_bad)
    s_ui_s = reward_module.predict_merge(sample_situations, sample_questions, sample_good)

    for weak, strong in zip(s_ui_w, s_ui_s):
        print("Weakener update:", weak)
        print("Strengthener update:", strong)
        print("-")
else:
    print("WARNING: Step 3-1 outputs or checkpoint-11000/ missing; skipping Step 3-3.")


In [ ]:
# STEP 3-4: Compute Delphi feedback and JS-Divergence reward (Equation 3).
if "s_ui_w" in globals() and "s_ui_s" in globals():
    P_jw = np.array(delphi_scorer.score_batch(s_ui_w))
    P_js = np.array(delphi_scorer.score_batch(s_ui_s))
    reward_module = load_reward_module()
    jsd_reward = reward_module.my_jensenshannon(P_jw, P_js, base=2, axis=1)
    print(jsd_reward)
else:
    print("WARNING: Step 3-3 outputs not found; skipping Step 3-4.")


In [ ]:
# STEP 3-5: Apply reward normalization (Equation 5).
required = [Path("out_dir_t5_large_answergen"), Path("checkpoint-11000")]

if "sample_situations" in globals() and "sample_questions" in globals() and require_paths(required, "Steps 2-2/2-3"):
    reward_module = load_reward_module()
    reward_module.model_nli = model_nli
    reward_module.tokenizer_nli = tokenizer_nli
    reward_module.model_dict_answer = reward_module.load_model_t5("out_dir_t5_large_answergen", cuda_devices=[0])
    reward_module.model_dict_fusion = reward_module.load_model_t5("checkpoint-11000", cuda_devices=[0])

    reward_model = reward_module.Reward(
        save_path="outputs/reward",
        device="cuda:0" if torch.cuda.is_available() else "cpu",
        batch_size=len(sample_situations),
        gain=GAIN,
        bias=BIAS,
    )
    reward_model.delphi_scorer = delphi_scorer
    normalized_rewards = reward_model.get_reward(sample_situations, sample_questions)
    print(normalized_rewards)
else:
    print("WARNING: Step 3-1 outputs or required checkpoints missing; skipping Step 3-5.")


**Phase 4: PPO Training**  
Purpose: Configure and run PPO training for ClarifyDelphi.


In [ ]:
# STEP 4-1: Set up PPO training environment.
sys.argv = ["notebook"]
args = get_args()
args.batch_size = 64
args.temperature = 0.7
args.top_p = 0.6
args.noptepochs = 4
args.kl_coef = 0.15
args.cliprange = 0.2
args.lam = 0.95
args.gamma = 1.0
args.total_episodes = 1000000
args.log_interval = 100
args.save_interval = 1000
args.eval_interval = 1000
args.output_dir = "outputs"
args.reward_dir = "outputs/reward"
args.model_dir = "outputs/model"
args.tensorboard_dir = "outputs/tensorboard"

num_gpus = torch.cuda.device_count()
print(f"CUDA available: {torch.cuda.is_available()} (GPUs: {num_gpus})")

for directory in ["outputs", "outputs/reward", "outputs/model", "outputs/tensorboard"]:
    os.makedirs(directory, exist_ok=True)


# STEP 4-2: Note on SocialChem-101 JSONL format.
Each line in the JSONL file must be a JSON object with key "prompt" containing a dict with key "text" set to the situation string.


In [ ]:
# STEP 4-2: Load PPO training data (SocialChem-101).
train_file = Path(args.dataset_train)
val_file = Path(args.dataset_val)

if require_paths([train_file, val_file, Path("output_t5_large_qgen")], "Step 4-2"):
    policy_tokenizer = T5Tokenizer.from_pretrained("output_t5_large_qgen")
    main_module = importlib.import_module("Code.main")
    PromptDataset = main_module.PromptDataset
    PromptCollator = main_module.PromptCollator

    train_dataset = PromptDataset(path=str(train_file))
    val_dataset = PromptDataset(path=str(val_file))

    prompt_collator = PromptCollator(tokenizer=policy_tokenizer)
    train_dataloader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, drop_last=True, collate_fn=prompt_collator)
    val_dataloader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=prompt_collator)

    print(f"Train examples: {len(train_dataset)}")
    print(f"Validation examples: {len(val_dataset)}")
else:
    print("ERROR: SocialChem-101 JSONL files missing. Generate them with the required prefix format before continuing.")
    train_dataloader = None
    val_dataloader = None


In [ ]:
# STEP 4-3: Initialize Policy, Value, and Reward models for PPO.
required = [Path("output_t5_large_qgen"), Path("out_dir_t5_large_answergen"), Path("checkpoint-11000")]

if train_dataloader is not None and torch.cuda.is_available() and require_paths(required, "Steps 2-1/2-2/2-3"):
    device = torch.device("cuda:0")
    ref_policy = Policy(model_name="output_t5_large_qgen", device=device)
    policy = Policy(model_name="output_t5_large_qgen", device=device)
    value = Value(model_type="output_t5_large_qgen", device=device)

    reward_module = load_reward_module()
    reward_module.model_nli = model_nli
    reward_module.tokenizer_nli = tokenizer_nli
    reward_module.model_dict_answer = reward_module.load_model_t5("out_dir_t5_large_answergen", cuda_devices=[0])
    reward_module.model_dict_fusion = reward_module.load_model_t5("checkpoint-11000", cuda_devices=[0])

    reward = reward_module.Reward(
        save_path="outputs/reward",
        device="cuda:0",
        batch_size=args.batch_size,
        gain=GAIN,
        bias=BIAS,
    )
    reward.delphi_scorer = delphi_scorer
    print("Loaded ref_policy, policy, value, reward.")
else:
    print("WARNING: Missing data/models or CUDA; skipping Step 4-3.")


In [ ]:
# STEP 4-4: Run PPO training loop (6000 steps).
if "policy" in globals() and train_dataloader is not None:
    args.total_steps = 6000
    optimizer = torch.optim.Adam(
        itertools.chain(policy.model.parameters(), value.model.parameters()),
        lr=1e-5,
        eps=1e-5,
    )
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=args.total_steps,
    )

    reward.set_reward_norm(dataloader=train_dataloader, policy=policy)

    main_module = importlib.import_module("Code.main")
    main_module.SummaryWriter = lambda: SummaryWriter(log_dir=args.tensorboard_dir)
    PPOTrainer = main_module.PPOTrainer

    trainer = PPOTrainer(
        params=args,
        policy=policy,
        ref_policy=ref_policy,
        value_model=value,
        score_model=reward,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        optimizer=optimizer,
        scheduler=scheduler,
    )

    for step_num in range(args.total_steps):
        try:
            trainer.step(step_num)
        except RuntimeError:
            torch.cuda.empty_cache()
            continue

    print("PPO training completed.")

    if Path("Data/delta-Clarify/test.tsv").exists():
        test_df = pd.read_csv("Data/delta-Clarify/test.tsv", sep="	").head(500)
        situations = test_df["situation"].astype(str).tolist()
        predictions = []
        for i in range(0, len(situations), 8):
            batch = situations[i:i+8]
            outputs = policy.sample(prompts=batch, top_p=0.6)
            predictions.extend(outputs["response/text"])
        pd.DataFrame({"situation": situations, "generated_question": predictions}).to_csv(
            "outputs/clarifydelphi_predictions.tsv", sep="	", index=False
        )
        print("Saved ClarifyDelphi predictions to outputs/clarifydelphi_predictions.tsv")
    else:
        print("WARNING: Data/delta-Clarify/test.tsv missing; skipping ClarifyDelphi predictions.")
else:
    print("WARNING: PPO prerequisites missing; skipping Step 4-4.")


In [ ]:
# STEP 4-5: Monitor training with TensorBoard.
log_dir = Path("outputs/tensorboard")
model_dir = Path("outputs/model")

if log_dir.exists():
    print("To view TensorBoard:")
    print("  In terminal: tensorboard --logdir outputs/tensorboard/")
    print("  In notebook: %load_ext tensorboard ; %tensorboard --logdir outputs/tensorboard/")
    print("Checkpoints:", [p.name for p in model_dir.glob("ckp_*.pth")])
else:
    print("WARNING: TensorBoard directory not found; run Step 4-4 first.")


**Phase 5: Baselines**  
Purpose: Generate baseline predictions for comparison with ClarifyDelphi.


In [ ]:
# STEP 5-1: Baseline inference: T5 fine-tuned.
required = [Path("output_t5_large_qgen"), Path("Data/delta-Clarify/test.tsv")]

if torch.cuda.is_available() and require_paths(required, "Step 2-1"):
    policy = Policy(model_name="output_t5_large_qgen", device="cuda:0")
    test_df = pd.read_csv("Data/delta-Clarify/test.tsv", sep="	")
    test_df = test_df.head(500)

    situations = test_df["situation"].astype(str).tolist()
    batch_size = 8
    predictions = []
    for i in range(0, len(situations), batch_size):
        batch = situations[i:i+batch_size]
        outputs = policy.sample(prompts=batch, top_p=0.6)
        predictions.extend(outputs["response/text"])

    out_df = pd.DataFrame({"situation": situations, "generated_question": predictions})
    out_df.to_csv("outputs/baseline_t5_finetuned_predictions.tsv", sep="	", index=False)
    print(out_df.head())
else:
    print("WARNING: CUDA or required files missing; skipping Step 5-1.")


In [ ]:
# STEP 5-2: Baseline: Discriminator (DeBERTa).
required = [Path("Data/delta-Clarify/train.tsv"), Path("output_t5_large_qgen"), Path("Data/delta-Clarify/test.tsv")]

if torch.cuda.is_available() and require_paths(required, "Step 2-1"):
    train_df = pd.read_csv("Data/delta-Clarify/train.tsv", sep="	")
    tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-large")
    model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-large", num_labels=2)

    questions = train_df["question"].astype(str).tolist()
    situations = train_df["situation"].astype(str).tolist()

    negatives = []
    for idx in range(len(questions)):
        neg_q = random.choice(questions)
        while neg_q == questions[idx]:
            neg_q = random.choice(questions)
        negatives.append(neg_q)

    class DiscriminatorDataset(Dataset):
        def __init__(self, situations, questions, labels):
            self.situations = situations
            self.questions = questions
            self.labels = labels

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            encoded = tokenizer(
                self.situations[idx],
                self.questions[idx],
                truncation=True,
                max_length=256,
                padding="max_length",
            )
            encoded["labels"] = self.labels[idx]
            return {k: torch.tensor(v) for k, v in encoded.items()}

    pos_labels = [1] * len(situations)
    neg_labels = [0] * len(situations)

    train_dataset = DiscriminatorDataset(situations + situations, questions + negatives, pos_labels + neg_labels)

    training_args = TrainingArguments(
        output_dir="outputs/deberta_discriminator",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        logging_strategy="epoch",
        save_strategy="no",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer,
    )

    trainer.train()

    device = torch.device("cuda:0")
    model.to(device)
    policy = Policy(model_name="output_t5_large_qgen", device=device)
    test_df = pd.read_csv("Data/delta-Clarify/test.tsv", sep="	").head(500)
    wh_words = ["what", "why", "how", "who", "where", "is", "when", "do", "did", "are", "does", "have", "was", "would"]

    selected_questions = []
    for situation in tqdm(test_df["situation"].astype(str).tolist()):
        candidates = []
        for wh in wh_words:
            prompt = f"{wh} {situation}"
            q = policy.sample(prompts=[prompt], top_p=0.6)["response/text"][0]
            candidates.append(q)

        encoded = tokenizer(
            [situation] * len(candidates),
            candidates,
            truncation=True,
            max_length=256,
            padding=True,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            logits = model(**encoded).logits
        probs = logits.softmax(dim=1)[:, 1]
        best_idx = int(torch.argmax(probs))
        selected_questions.append(candidates[best_idx])

    out_df = pd.DataFrame({"situation": test_df["situation"].astype(str).tolist(), "generated_question": selected_questions})
    out_df.to_csv("outputs/baseline_discriminator_predictions.tsv", sep="	", index=False)
    print(out_df.head())
else:
    print("WARNING: CUDA or required files missing; skipping Step 5-2.")


In [ ]:
# STEP 5-3: Baseline: Pipeline and Pipeline_NLI.
required = [
    Path("output_t5_large_qgen"),
    Path("out_dir_t5_large_answergen"),
    Path("checkpoint-11000"),
    Path("Data/delta-Clarify/test.tsv"),
]

if torch.cuda.is_available() and require_paths(required, "Steps 2-1/2-2/2-3"):
    reward_module = load_reward_module()
    reward_module.model_nli = model_nli
    reward_module.tokenizer_nli = tokenizer_nli
    reward_module.model_dict_answer = reward_module.load_model_t5("out_dir_t5_large_answergen", cuda_devices=[0])
    reward_module.model_dict_fusion = reward_module.load_model_t5("checkpoint-11000", cuda_devices=[0])

    policy = Policy(model_name="output_t5_large_qgen", device="cuda:0")
    test_df = pd.read_csv("Data/delta-Clarify/test.tsv", sep="	").head(500)
    situations = test_df["situation"].astype(str).tolist()
    wh_words = ["what", "why", "how", "who", "where", "is", "when", "do", "did", "are", "does", "have", "was", "would"]

    def generate_answers_no_nli(situations, questions, judgements):
        tokenizer = reward_module.model_dict_answer["tokenizer"]
        model = reward_module.model_dict_answer["model"]
        model_inputs, uts = [], []
        for jud, sit, q in zip(judgements, situations, questions):
            for ut in ["weakener", "strengthener"]:
                if not sit.endswith("."):
                    sit = sit + "."
                model_inputs.append("answer: " + jud + " " + sit + " TYPE: " + ut + " QUESTION: " + q)
                uts.append(ut)
        inputs = tokenizer(model_inputs, return_tensors="pt", padding=True).to(
            reward_module.model_dict_answer["cuda_device"]
        )
        response_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            do_sample=True,
            max_length=200,
            top_p=0.6,
            pad_token_id=tokenizer.pad_token_id,
        )
        pred = tokenizer.batch_decode(response_ids, skip_special_tokens=True)
        bad, good = [], []
        for answer, ut in zip(pred, uts):
            if ut == "weakener":
                bad.append(answer)
            else:
                good.append(answer)
        return bad, good

    pipeline_questions = []
    pipeline_nli_questions = []

    for situation in tqdm(situations):
        candidates = []
        for wh in wh_words:
            prompt = f"{wh} {situation}"
            q = policy.sample(prompts=[prompt], top_p=0.6)["response/text"][0]
            candidates.append(q)

        _, judgements = delphi_scorer.generate_batch([situation] * len(candidates))

        # Pipeline (no NLI filtering)
        bad_raw, good_raw = generate_answers_no_nli([situation] * len(candidates), candidates, judgements)
        s_w = reward_module.predict_merge([situation] * len(candidates), candidates, bad_raw)
        s_s = reward_module.predict_merge([situation] * len(candidates), candidates, good_raw)
        P_w = np.array(delphi_scorer.score_batch(s_w))
        P_s = np.array(delphi_scorer.score_batch(s_s))
        jsd_scores = reward_module.my_jensenshannon(P_w, P_s, base=2, axis=1)
        pipeline_questions.append(candidates[int(np.argmax(jsd_scores))])

        # Pipeline_NLI
        bad_f, good_f = reward_module.get_answers_from_model_batch_t5(
            [situation] * len(candidates), candidates, judgements
        )
        s_w_nli = reward_module.predict_merge([situation] * len(candidates), candidates, bad_f)
        s_s_nli = reward_module.predict_merge([situation] * len(candidates), candidates, good_f)
        P_w_nli = np.array(delphi_scorer.score_batch(s_w_nli))
        P_s_nli = np.array(delphi_scorer.score_batch(s_s_nli))
        jsd_scores_nli = reward_module.my_jensenshannon(P_w_nli, P_s_nli, base=2, axis=1)
        pipeline_nli_questions.append(candidates[int(np.argmax(jsd_scores_nli))])

    pd.DataFrame({"situation": situations, "generated_question": pipeline_questions}).to_csv(
        "outputs/baseline_pipeline_predictions.tsv", sep="	", index=False
    )
    pd.DataFrame({"situation": situations, "generated_question": pipeline_nli_questions}).to_csv(
        "outputs/baseline_pipeline_nli_predictions.tsv", sep="	", index=False
    )
else:
    print("WARNING: CUDA or required files missing; skipping Step 5-3.")


In [ ]:
# STEP 5-4: Baseline: Why-baseline.
required = [Path("output_t5_large_qgen"), Path("Data/delta-Clarify/test.tsv")]

if torch.cuda.is_available() and require_paths(required, "Step 2-1"):
    policy = Policy(model_name="output_t5_large_qgen", device="cuda:0")
    test_df = pd.read_csv("Data/delta-Clarify/test.tsv", sep="	").head(500)
    situations = test_df["situation"].astype(str).tolist()

    predictions = []
    for situation in situations:
        prompt = f"why {situation}"
        question = policy.sample(prompts=[prompt], top_p=0.6)["response/text"][0]
        predictions.append(question)

    out_df = pd.DataFrame({"situation": situations, "generated_question": predictions})
    out_df.to_csv("outputs/baseline_why_predictions.tsv", sep="	", index=False)
    print(out_df.head())
else:
    print("WARNING: CUDA or required files missing; skipping Step 5-4.")


**Phase 6: Evaluation**  
Purpose: Run automatic evaluations and record required notes about human evaluation.


In [ ]:
# STEP 6-1: Automatic evaluation: Informativeness (SQuAD 2.0 QA model).
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2", handle_impossible_answer=True)

prediction_files = {
    "clarifydelphi": "outputs/clarifydelphi_predictions.tsv",
    "baseline_t5": "outputs/baseline_t5_finetuned_predictions.tsv",
    "baseline_discriminator": "outputs/baseline_discriminator_predictions.tsv",
    "baseline_pipeline": "outputs/baseline_pipeline_predictions.tsv",
    "baseline_pipeline_nli": "outputs/baseline_pipeline_nli_predictions.tsv",
    "baseline_why": "outputs/baseline_why_predictions.tsv",
}

informativeness = {}
for name, path in prediction_files.items():
    if not Path(path).exists():
        print(f"WARNING: Missing {path}; skipping {name}")
        continue
    df = pd.read_csv(path, sep="	")
    total = len(df)
    unanswerable = 0
    for _, row in df.iterrows():
        result = qa_pipeline(question=row["generated_question"], context=row["situation"])
        if result.get("answer", "").strip() == "":
            unanswerable += 1
    informativeness[name] = 100.0 * unanswerable / max(total, 1)

summary_df = pd.DataFrame(
    [{"model": k, "informativeness_unanswerable_pct": v} for k, v in informativeness.items()]
)
print(summary_df)


In [ ]:
# STEP 6-2: Automatic evaluation: BERTScore.
if Path("Data/delta-Clarify/test.tsv").exists():
    test_df = pd.read_csv("Data/delta-Clarify/test.tsv", sep="	")
    grouped = test_df.groupby("situation")["question"].apply(list).to_dict()

    bertscore_results = {}
    for name, path in prediction_files.items():
        if not Path(path).exists():
            continue
        preds_df = pd.read_csv(path, sep="	")
        scores = []
        for _, row in preds_df.iterrows():
            refs = grouped.get(row["situation"], [])
            if not refs:
                continue
            cand = row["generated_question"]
            best = None
            for ref in refs:
                _, _, f1 = bert_score([cand], [ref], lang="en", rescale_with_baseline=True)
                score = float(f1.mean())
                best = score if best is None else max(best, score)
            if best is not None:
                scores.append(best)
        bertscore_results[name] = float(np.mean(scores)) if scores else float("nan")

    print(pd.DataFrame([{"model": k, "bertscore_f1": v} for k, v in bertscore_results.items()]))
else:
    print("WARNING: Data/delta-Clarify/test.tsv missing; skipping Step 6-2.")


In [ ]:
# STEP 6-3: Automatic evaluation: BLEU-4 and ROUGE-L.
sc_test_path = Path("Data/delta-Clarify-silver/sc_test_questions_gpt.tsv")

if sc_test_path.exists() and Path("out_dir_t5_large_answergen").exists():
    test_df = pd.read_csv(sc_test_path, sep="	")
    tokenizer = T5Tokenizer.from_pretrained("out_dir_t5_large_answergen")
    model = T5ForConditionalGeneration.from_pretrained("out_dir_t5_large_answergen")

    inputs = (
        "answer: "
        + test_df["Hypothesis"].astype(str)
        + " TYPE: "
        + test_df["UpdateType"].astype(str)
        + " QUESTION: "
        + test_df["question_davinci"].astype(str)
    ).tolist()

    references = test_df["Update"].astype(str).tolist()

    predictions = []
    for inp in tqdm(inputs):
        encoded = tokenizer(inp, return_tensors="pt", truncation=True)
        output_ids = model.generate(**encoded, max_length=128)
        predictions.append(tokenizer.decode(output_ids[0], skip_special_tokens=True))

    smoothie = SmoothingFunction().method4
    bleu = corpus_bleu(
        [[ref.split()] for ref in references],
        [pred.split() for pred in predictions],
        smoothing_function=smoothie,
    )

    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_scores = [scorer.score(ref, pred)["rougeL"].fmeasure for ref, pred in zip(references, predictions)]
    rouge_l = float(np.mean(rouge_scores)) * 100

    print({"BLEU-4": bleu * 100, "ROUGE-L": rouge_l})
    print("Baseline comparisons: delta-SOCIAL T5-large BLEU=4.22 ROUGE=14.94; delta-SOCIAL GPT2-XL BLEU=12.16 ROUGE=18.77")
else:
    print("WARNING: sc_test_questions_gpt.tsv missing or model not trained; skipping Step 6-3.")


In [ ]:
# STEP 6-4: Automatic evaluation: Average JSD and Judgment Flip Rate.
if Path("Data/delta-Clarify/test.tsv").exists() and Path("out_dir_t5_large_answergen").exists() and Path("checkpoint-11000").exists():
    reward_module = load_reward_module()
    reward_module.model_nli = model_nli
    reward_module.tokenizer_nli = tokenizer_nli
    reward_module.model_dict_answer = reward_module.load_model_t5("out_dir_t5_large_answergen", cuda_devices=[0])
    reward_module.model_dict_fusion = reward_module.load_model_t5("checkpoint-11000", cuda_devices=[0])

    def compute_jsd_and_flip(df):
        situations = df["situation"].astype(str).tolist()
        questions = df["generated_question"].astype(str).tolist()
        _, judgements = delphi_scorer.generate_batch(situations)
        bad, good = reward_module.get_answers_from_model_batch_t5(situations, questions, judgements)
        s_w = reward_module.predict_merge(situations, questions, bad)
        s_s = reward_module.predict_merge(situations, questions, good)
        P_w = np.array(delphi_scorer.score_batch(s_w))
        P_s = np.array(delphi_scorer.score_batch(s_s))
        jsd = reward_module.my_jensenshannon(P_w, P_s, base=2, axis=1)
        flips = (np.argmax(P_w, axis=1) != np.argmax(P_s, axis=1)).astype(float)
        return float(np.mean(jsd)), float(np.mean(flips)) * 100

    results = []
    for name, path in prediction_files.items():
        if not Path(path).exists():
            continue
        df = pd.read_csv(path, sep="	")
        avg_jsd, flip_rate = compute_jsd_and_flip(df)
        results.append({"model": name, "avg_jsd": avg_jsd, "flip_rate_pct": flip_rate})

    print(pd.DataFrame(results))
    print("Expected: ClarifyDelphi avg JSD=0.191, Flip Rate=25%; pipeline_nli avg JSD=0.259, Flip Rate=33%.")
else:
    print("WARNING: Required models or data missing; skipping Step 6-4.")


# STEP 6-5: Human evaluation note.
Direct MTurk human evaluation requires recruiting annotators and cannot be automated in this notebook. A GPT-4-based automatic approximation can be used as a substitute; if desired, add a GPT-4 API call that rates grammaticality, relevance, informativeness, and defeasibility and saves results to outputs/gpt4_human_eval_substitute.tsv.
